# Image Processing in Python — lecture code

Every snippet from *DIP-Python-Introduction-NDU.pptx*, made runnable and keyed
to the slide it appears on. Run the cells in order during the lecture, or hand
the notebook to students afterwards.

Nothing to install in Colab. Locally:

```
pip install numpy opencv-python scikit-image matplotlib
```

## Slide 8 — Getting set up

The opening cell. Worth typing from scratch in front of the class.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage import data

img = data.camera()

print(img.shape)          # (512, 512)
print(img.dtype)          # uint8
print(img.min(), img.max())

plt.imshow(img, cmap='gray', vmin=0, vmax=255)
plt.axis('off')
plt.show()

## Slide 10 — An image is just a NumPy array

In [ ]:
print(img.shape)         # (512, 512)
print(img.dtype)         # uint8
print(img.ndim)          # 2  -> grayscale
print(img[213, 202])     # 43   a single pixel
print(img.mean())        # 129.06  one scalar over 262,144 pixels

# the 8 x 8 patch shown on the slide
print(img[210:218, 200:208])

## Slide 11 — uint8 and float images

A `uint8` image runs 0 to 255. Convert it to float and you choose the
convention yourself — almost always 0 to 1 — and nothing in the array records
that choice. A NumPy dtype, like a C++ type, tells you how the bytes are
stored, not what they mean.

In [ ]:
# uint8 -> float in [0, 1]
f = img.astype(np.float64) / 255.0
print('float range :', f.min(), f.max())

# float -> uint8.  The clip does nothing on this line, but any operation on f
# can push values outside [0, 1], and the cast wraps instead of saturating.
u = np.clip(f * 255, 0, 255).astype(np.uint8)
print('back to uint8:', u.dtype, u.min(), u.max())

# rescale an array whose range you do not know in advance
x = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(f))))   # 0 .. ~11
print('unknown range:', x.min().round(2), x.max().round(2))

g = cv2.normalize(x, None, 0, 1, cv2.NORM_MINMAX, cv2.CV_64F)
print('after normalize:', g.min(), g.max())

`cv2.CV_64F` on that last line is not optional. It is OpenCV's type code for a
64-bit float (a C++ `double`, a NumPy `float64`), and it sets the type of the output.
Leave it out and the output keeps the input's type, so a 0–1 rescale of an integer
image collapses to zeros and ones.

In [ ]:
row = (300, slice(200, 206))          # a slice that straddles mid-grey
print('input           ', img[row])

bad  = cv2.normalize(img, None, 0, 1, cv2.NORM_MINMAX)                # dtype omitted
good = cv2.normalize(img, None, 0, 1, cv2.NORM_MINMAX, cv2.CV_64F)
print('no dtype        ', bad[row], bad.dtype, ' <- collapsed to 0s and 1s')
print('with cv2.CV_64F ', good[row].round(3), good.dtype)

# NORM_MINMAX must be given explicitly - the default is NORM_L2
print('\ndefault norm_type is NORM_L2:',
      cv2.normalize(np.float64([[3, 4]]), None).ravel(), '<- unit length, not a range')

## Slide 12 — Two indexing conventions

Arrays are `(row, column)`. OpenCV points are `(x, y)` and sizes are `(width, height)`.

In [ ]:
y, x = 213, 202
print('img[y, x] =', img[y, x])          # row first

canvas = img.copy()
cv2.circle(canvas, (x, y), 25, 255, 2)   # x first
small = cv2.resize(img, (128, 64))       # width first
print('resize((128, 64)) gives shape', small.shape)   # (64, 128)

plt.imshow(canvas, cmap='gray', vmin=0, vmax=255); plt.axis('off'); plt.show()

## Slide 13 — Cropping is slicing

In [ ]:
roi   = img[60:220, 180:340]
top   = img[:100, :]
right = img[:, -100:]
flip  = img[::-1, :]

for name, a in [('roi', roi), ('top', top), ('right', right), ('flip', flip)]:
    print(f'{name:6s} {a.shape}')

**Views versus copies.** The one real surprise. A slice is a pointer into the
original buffer, not a copy of it — writing through the slice writes into the
image it came from. Add `.copy()` when you do not want that.

In [ ]:
demo = img.copy()
view = demo[60:220, 180:340]     # a view into demo
view[:] = 0                      # writes straight through

safe = img[60:220, 180:340].copy()
safe[:] = 0                      # img is untouched

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(demo, cmap='gray', vmin=0, vmax=255); ax[0].set_title('after view[:] = 0')
ax[1].imshow(img,  cmap='gray', vmin=0, vmax=255); ax[1].set_title('after safe[:] = 0')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

## Slide 15 — imread returns BGR, not RGB

There is no file to read here, so the BGR array `cv2.imread` would hand back is
built from a skimage RGB image. The point is identical.

In [ ]:
rgb_true = data.coffee()                     # skimage gives RGB
bgr      = cv2.cvtColor(rgb_true, cv2.COLOR_RGB2BGR)   # what imread returns

# bgr = cv2.imread('photo.jpg')              # the real call
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)   # the fix

fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
ax[0].imshow(bgr); ax[0].set_title('plt.imshow(bgr)  — wrong')
ax[1].imshow(rgb); ax[1].set_title('after cvtColor  — right')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

# grayscale sidesteps the whole issue
# g = cv2.imread('photo.jpg', cv2.IMREAD_GRAYSCALE)

## Slide 16 — The display helper

Paste this into the first cell of every notebook this term.

`uint8` images are pinned to 0–255 so that contrast changes stay visible (Slide 17).
Anything else — floats, 16-bit — is left to autoscale, because its range is not
known in advance.

In [ ]:
def show(*imgs, titles=None, size=4):
    n = len(imgs)
    fig, ax = plt.subplots(1, n, figsize=(size*n, size))
    ax = np.atleast_1d(ax).ravel()
    for i, (a, im) in enumerate(zip(ax, imgs)):
        if im.ndim == 2:
            if im.dtype == np.uint8:
                a.imshow(im, cmap='gray', vmin=0, vmax=255)
            else:
                a.imshow(im, cmap='gray')      # float or 16-bit: autoscale
        else:
            a.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        if titles:
            a.set_title(titles[i])
        a.axis('off')
    plt.tight_layout()
    plt.show()


show(img,
     cv2.GaussianBlur(img, (9, 9), 0),
     cv2.Canny(img, 100, 200),
     titles=['original', 'Gaussian 9x9', 'Canny'], size=3)

## Slide 17 — Always pin vmin and vmax

In [ ]:
dark = (img * 0.35).astype(np.uint8)

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(dark, cmap='gray')
ax[0].set_title("cmap='gray'  — autoscaled, lies")
ax[1].imshow(dark, cmap='gray', vmin=0, vmax=255)
ax[1].set_title('vmin=0, vmax=255  — true')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

print('the array really spans', dark.min(), 'to', dark.max())

## Slide 19 — Point operations are array arithmetic

In [ ]:
neg = 255 - img                                 # negative
bw  = ((img > 128) * 255).astype(np.uint8)      # threshold to black and white

f   = img / 255.0                               # gamma correction
out = ((f ** 0.4) * 255).astype(np.uint8)

show(img, neg, bw, out,
     titles=['original', '255 - img', 'img > 128', 'gamma 0.4'], size=3)

### The range trap

The `f = img / 255.0` on the gamma line above is not decoration. Drop it and the
same expression produces noise.

`img` runs 0–255, so `img ** 0.4` runs 0–9.175, and multiplying by 255 again puts the
top of the scale at 2339.7. The cast to `uint8` takes that modulo 256, so most pixels
wrap round several times.

The constant is what is missing, not the clip: clipping alone gives a pure white image,
because 2339 is far past the ceiling.

In [ ]:
out1 = ((img ** 0.4) * 255).astype(np.uint8)      # the /255 forgotten

p = img ** 0.4
print('img ** 0.4        ->', p.min().round(3), 'to', p.max().round(3), ' (not 0..1)')
print('* 255             ->', (p * 255).max().round(1))
print('after astype      ->', out1.min(), out1.max(),
      ' <- any wrapped image prints 0 255')
print('pixels that wrapped:', int((p * 255 >= 256).sum()), 'of', img.size)

show(img, out, out1,
     titles=['original', 'gamma 0.4', 'gamma 0.4 - forgot to normalize'], size=4)

Three ways to get it right. The first two are the *same* transformation — a pure
gamma, with a mapping that does not depend on the image. The third rescales by whatever
range the data happens to occupy, which is a different operation whenever the image does
not already span 0–255.

All three round with `np.rint` before the cast. `.astype` on its own truncates, which
leaves neighbouring methods disagreeing by one level for no reason.

In [ ]:
# 1. normalize first  (what `out` does)
out_a = np.rint(np.clip((img / 255.0) ** 0.4 * 255, 0, 255)).astype(np.uint8)

# 2. stay on 0..255 and supply the constant the range demands
c = 255 / (255 ** 0.4)
print('c =', round(c, 3))
out_b = np.rint(np.clip(c * img ** 0.4, 0, 255)).astype(np.uint8)

# 3. remap whatever comes out.  No cast and no *255: ** promotes to float,
#    and a positive scale factor cancels inside a min-max rescale.
s = img ** 0.4
out_c = np.rint(cv2.normalize(s, None, 0, 255, cv2.NORM_MINMAX)).astype(np.uint8)

for name, v in [('fixed constant', out_b), ('min-max remap', out_c)]:
    print(f'{name:15} max difference from (1):',
          np.abs(v.astype(int) - out_a.astype(int)).max())

show(out_a, out_b, out_c,
     titles=['normalize first', f'constant c = {c:.2f}', 'min-max remap'], size=4)

They agree here only because the cameraman already spans the full 0–255 range.
On an image that does not, min-max adds a contrast stretch on top of the gamma, and the
brightness change that the gamma was supposed to show is partly erased.

In [ ]:
crop = img[0:120, 250:400]          # spans 10..211, not the full range
print('crop range:', crop.min(), '..', crop.max(), '\n')
print(f"{'gamma':>6} {'fixed c':>10} {'min-max':>10}   (mean brightness of the result)")
for g in (0.4, 1.0, 2.5):
    a = np.clip((crop / 255.0) ** g * 255, 0, 255).astype(np.uint8)
    b = cv2.normalize((crop.astype(np.float64) ** g), None, 0, 255,
                      cv2.NORM_MINMAX).astype(np.uint8)
    print(f'{g:>6} {a.mean():>10.1f} {b.mean():>10.1f}')

print('\nfixed c : brightness tracks gamma - the point of the demonstration')
print('min-max : every output restretched to 0..255, so the effect is flattened')

## Slide 20 — Integer arithmetic wraps

Ask the room what `200 + 100` gives before running this. It is the same
overflow rule as an `unsigned char` in C++: the result is taken mod 256.

In [ ]:
p = np.uint8([[200]])                  # a one-pixel image

print('p + 100          =', (p + np.uint8(100))[0, 0])   # 44   wraps
print('cv2.add(p, 100)  =', cv2.add(p, 100)[0, 0])       # 255  saturates

widened = np.clip(p.astype(np.int16) + 100, 0, 255).astype(np.uint8)
print('widen and clip   =', widened[0, 0])

# the same thing across the whole intensity range
r = np.arange(256, dtype=np.uint8)
plt.plot(r, r + np.uint8(100), label='img + 100  (wraps)')
plt.plot(r, cv2.add(r.reshape(-1, 1), 100).ravel(), label='cv2.add  (saturates)')
plt.xlabel('input'); plt.ylabel('output'); plt.legend(frameon=False); plt.show()

**The same trap through a different door.** The *type* of an exponent decides
whether NumPy promotes to float. A float exponent promotes; an integer one does not,
so the arithmetic stays in `uint8` and wraps.

In [ ]:
for e in (2, 2.0, 0.4):
    r = img ** e
    flag = '   <-- stayed uint8, wrapped' if r.dtype == np.uint8 else ''
    print(f'img ** {e!r:4} -> {str(r.dtype):8} max {r.max():>9.1f}{flag}')

print('\n255 ** 2 =', 255**2, '- nowhere near fitting in a byte')

## Slide 21 — Lookup tables

Every point transformation in the course should end up here.

In [ ]:
moon = data.moon()

r   = np.arange(256)
lut = np.clip((r - 80) * 255.0 / 80, 0, 255).astype(np.uint8)
out = cv2.LUT(moon, lut)

fig = plt.figure(figsize=(11, 3.4))
a = fig.add_subplot(1, 3, 1); a.plot(lut); a.set_xlim(0, 255); a.set_ylim(0, 255)
a.set_xlabel('r'); a.set_ylabel('s'); a.set_title('the lookup table')
for k, (im, t) in enumerate([(moon, 'before'), (out, 'after cv2.LUT')]):
    b = fig.add_subplot(1, 3, k+2)
    b.imshow(im, cmap='gray', vmin=0, vmax=255); b.set_title(t); b.axis('off')
plt.tight_layout(); plt.show()

## Slide 22 — Histograms

In [ ]:
h1 = np.bincount(moon.ravel(), minlength=256)              # one bin per level
h2 = cv2.calcHist([moon], [0], None, [256], [0, 256])      # OpenCV

print('identical:', np.allclose(h1, h2.ravel()))

fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
ax[0].imshow(moon, cmap='gray', vmin=0, vmax=255); ax[0].axis('off'); ax[0].set_title('moon()')
ax[1].bar(np.arange(256), h1, width=1.0); ax[1].set_xlim(0, 255)
ax[1].set_title('bincount — full 0..255 range')
ax[2].hist(moon.ravel())                                    # 10 bins by default
ax[2].set_title('plt.hist — data range, 10 bins')
plt.tight_layout(); plt.show()

## Slide 24 — Where each job lives

| Task | Python |
|---|---|
| read / write a file | `cv2.imread` / `cv2.imwrite` |
| display in a notebook | `plt.imshow`, or the `show` helper above |
| uint8 to float in [0, 1] | `img.astype(np.float64) / 255` |
| rescale an unknown range to [0, 1] | `cv2.normalize(..., cv2.NORM_MINMAX)` |
| negative | `255 - img` |
| crop | `img[r0:r1, c0:c1]` |
| histogram | `np.bincount(img.ravel(), minlength=256)` |
| apply a point transformation | `cv2.LUT` |
| histogram equalization | `cv2.equalizeHist` |
| local (adaptive) equalization | `cv2.createCLAHE` |
| filter with a kernel | `cv2.filter2D` |
| automatic threshold | `cv2.threshold(..., cv2.THRESH_OTSU)` |
| measure labelled regions | `skimage.measure.regionprops` |

## Slide 25 — The five that cost the most time

1. **BGR, not RGB** — `cvtColor` on display, always.
2. **Integer arithmetic wraps** — `cv2.add`, or widen and clip.
3. **Matplotlib autoscales** — pin `vmin=0, vmax=255` on every uint8 image.
4. **dtype does not imply a range** — print it and check.
5. **Slices are views** — writing into a region of interest writes into the original.